# Chapter 1: LLM API Comparison for Marketing Technology

**From *Mastering Agentic AI for Marketing Technology* by Pushparajan Ramar**

---

Modern martech stacks increasingly rely on Large Language Models (LLMs) to generate copy, personalise
messages, and power conversational interfaces. In this notebook we explore the three dominant LLM
providers — **OpenAI GPT-4.1**, **Anthropic Claude**, and **Google Gemini** — through a unified
OpenAI-compatible client pattern.

### Why a unified interface matters

Marketing teams need to:
- **A/B test** outputs across models to find the best copy.
- **Fail over** seamlessly when one provider has an outage.
- **Benchmark cost and latency** without rewriting integration code.

The OpenAI Python SDK now supports a `base_url` parameter that lets you point the same client at
any OpenAI-compatible endpoint — including Anthropic’s and Google’s compatibility layers.

### What you will learn

1. How to configure the OpenAI client for three different providers.
2. How to generate marketing copy (email subject lines) with each model.
3. How to compare outputs side-by-side.

## 1 — Environment Setup

We load API keys from a `.env` file (never hard-code secrets!) and set a `USE_MOCK` flag so the
notebook runs even without live credentials.

In [ ]:
# --- Environment & mock configuration ---
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env in the repo root

USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"

print(f"USE_MOCK = {USE_MOCK}")
if USE_MOCK:
    print("Running with mock responses. Set USE_MOCK_APIS=false and provide API keys for live calls.")

## 2 — Provider Configuration

Each provider needs three things: a **base URL**, an **API key**, and a **model identifier**.
The table below summarises the endpoints:

| Provider  | Base URL                                    | Model example        |
|-----------|---------------------------------------------|----------------------|
| OpenAI    | `https://api.openai.com/v1`                 | `gpt-4.1`            |
| Anthropic | `https://api.anthropic.com/v1`              | `claude-sonnet-4-20250514`  |
| Google    | `https://generativelanguage.googleapis.com/v1beta/openai` | `gemini-2.0-flash`   |

In [ ]:
# --- Provider definitions ---

PROVIDERS = {
    "openai": {
        "base_url": "https://api.openai.com/v1",
        "api_key_env": "OPENAI_API_KEY",
        "model": "gpt-4.1",
        "display_name": "OpenAI GPT-4.1",
    },
    "anthropic": {
        "base_url": "https://api.anthropic.com/v1",
        "api_key_env": "ANTHROPIC_API_KEY",
        "model": "claude-sonnet-4-20250514",
        "display_name": "Anthropic Claude Sonnet",
    },
    "google": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai",
        "api_key_env": "GOOGLE_API_KEY",
        "model": "gemini-2.0-flash",
        "display_name": "Google Gemini 2.0 Flash",
    },
}

print("Configured providers:")
for key, cfg in PROVIDERS.items():
    has_key = bool(os.getenv(cfg["api_key_env"]))
    status = "API key found" if has_key else "no key — will use mock"
    print(f"  {cfg['display_name']:30s} — {status}")

## 3 — Mock Response Layer

When `USE_MOCK` is `True` we return realistic pre-built responses so the notebook is runnable
without any API keys. The mock layer mirrors the structure of a real `ChatCompletion` response.

In [ ]:
# --- Mock responses ---
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class MockMessage:
    role: str = "assistant"
    content: str = ""


@dataclass
class MockChoice:
    index: int = 0
    message: MockMessage = field(default_factory=MockMessage)
    finish_reason: str = "stop"


@dataclass
class MockUsage:
    prompt_tokens: int = 42
    completion_tokens: int = 38
    total_tokens: int = 80


@dataclass
class MockChatCompletion:
    id: str = "mock-completion-001"
    object: str = "chat.completion"
    model: str = "mock-model"
    choices: List[MockChoice] = field(default_factory=list)
    usage: MockUsage = field(default_factory=MockUsage)


MOCK_SUBJECT_LINES = {
    "openai": (
        "1. \"Your Q2 Growth Blueprint Is Inside — Open Now\"\n"
        "2. \"We Analyzed 10K Campaigns So You Don’t Have To\"\n"
        "3. \"[First Name], Your Competitors Read This Last Week\""
    ),
    "anthropic": (
        "1. \"The Data-Backed Playbook for Q2 Pipeline Growth\"\n"
        "2. \"3 Campaign Tweaks That Lifted CTR by 40%\"\n"
        "3. \"[First Name], Here’s What Top Marketers Do Differently\""
    ),
    "google": (
        "1. \"Unlock Your Q2 Marketing Edge — Free Report\"\n"
        "2. \"Steal These 3 Strategies from Award-Winning Campaigns\"\n"
        "3. \"[First Name], Your Personalised Growth Plan Awaits\""
    ),
}


def mock_completion(provider_key: str, model: str) -> MockChatCompletion:
    """Return a realistic mock ChatCompletion for the given provider."""
    content = MOCK_SUBJECT_LINES.get(provider_key, "Mock subject line")
    return MockChatCompletion(
        id=f"mock-{provider_key}-001",
        model=model,
        choices=[MockChoice(message=MockMessage(content=content))],
    )


print("Mock layer ready.")

## 4 — Unified `call_llm` Helper

This is the key abstraction. A single function creates an `OpenAI` client pointed at the right
`base_url`, sends the prompt, and returns the completion. When `USE_MOCK` is on, the live call is
replaced with our mock layer.

In [ ]:
# --- Unified LLM caller ---
from openai import OpenAI


def call_llm(
    provider_key: str,
    messages: list,
    temperature: float = 0.7,
    max_tokens: int = 256,
):
    """
    Call any OpenAI-compatible LLM using the unified client pattern.

    Parameters
    ----------
    provider_key : str
        One of 'openai', 'anthropic', 'google'.
    messages : list[dict]
        Chat-format messages (role / content dicts).
    temperature : float
        Sampling temperature.
    max_tokens : int
        Maximum tokens to generate.

    Returns
    -------
    ChatCompletion (or MockChatCompletion)
    """
    cfg = PROVIDERS[provider_key]

    if USE_MOCK:
        print(f"  [{cfg['display_name']}] → returning mock response")
        return mock_completion(provider_key, cfg["model"])

    api_key = os.getenv(cfg["api_key_env"], "")
    if not api_key:
        print(f"  [{cfg['display_name']}] → no API key, falling back to mock")
        return mock_completion(provider_key, cfg["model"])

    client = OpenAI(
        base_url=cfg["base_url"],
        api_key=api_key,
    )

    response = client.chat.completions.create(
        model=cfg["model"],
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    print(f"  [{cfg['display_name']}] → live response ({response.usage.total_tokens} tokens)")
    return response


print("call_llm() ready.")

## 5 — Martech Use Case: Email Subject Line Generation

Let’s put the unified interface to work. We’ll ask each model to generate three email subject
lines for a B2B SaaS marketing campaign. The prompt includes brand context and audience
information — exactly how a marketing automation platform would call an LLM.

In [ ]:
# --- Prompt definition ---

SYSTEM_PROMPT = (
    "You are an expert B2B email copywriter. You write concise, compelling subject lines "
    "that maximise open rates while maintaining a professional tone. Avoid spam-trigger "
    "words. Keep each subject line under 60 characters."
)

USER_PROMPT = (
    "Generate 3 email subject lines for the following campaign:\n\n"
    "Product: MarketFlow — an AI-powered marketing automation platform\n"
    "Audience: VP/Director of Marketing at mid-market SaaS companies (200–2000 employees)\n"
    "Campaign goal: Drive downloads of our new Q2 Benchmark Report\n"
    "Tone: Authoritative but approachable\n\n"
    "Return only the numbered list of subject lines."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

print("Prompt ready. Calling all three providers...\n")

In [ ]:
# --- Call each provider and collect results ---

results = {}

for provider_key in PROVIDERS:
    response = call_llm(provider_key, messages)
    results[provider_key] = response.choices[0].message.content

print("\nAll calls complete.")

## 6 — Side-by-Side Comparison

Comparing outputs from different models is a core martech workflow. In production you would
feed these into an A/B testing framework; here we simply display them.

In [ ]:
# --- Display results ---

for provider_key, content in results.items():
    display_name = PROVIDERS[provider_key]["display_name"]
    print(f"{'=' * 60}")
    print(f"  {display_name}")
    print(f"{'=' * 60}")
    print(content)
    print()

## 7 — Building a Simple Evaluation Function

In real martech scenarios, you need a way to programmatically evaluate outputs. Below is a
lightweight scorer that checks for common subject-line best practices.

In [ ]:
# --- Simple subject-line evaluator ---

import re

SPAM_WORDS = {"free", "act now", "limited time", "buy now", "click here", "urgent", "winner"}


def score_subject_lines(text: str) -> dict:
    """
    Score a set of generated subject lines on basic quality criteria.

    Returns a dict with per-line scores and an overall average.
    """
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    scores = []

    for line in lines:
        # Strip leading number/bullet
        clean = re.sub(r'^\d+[\.\)\-]\s*', '', line).strip().strip('"')
        score = 0
        reasons = []

        # Length check: ideal 30-60 characters
        length = len(clean)
        if 30 <= length <= 60:
            score += 3
            reasons.append(f"good length ({length} chars)")
        elif length < 30:
            score += 1
            reasons.append(f"too short ({length} chars)")
        else:
            score += 1
            reasons.append(f"too long ({length} chars)")

        # Personalisation token
        if "[first name]" in clean.lower() or "{first_name}" in clean.lower():
            score += 2
            reasons.append("personalisation token")

        # Spam-word check
        lower = clean.lower()
        found_spam = [w for w in SPAM_WORDS if w in lower]
        if not found_spam:
            score += 2
            reasons.append("no spam words")
        else:
            reasons.append(f"spam words found: {found_spam}")

        # Urgency / curiosity hook
        hooks = ["?", "...", "\u2014", ":", "how", "why", "what", "inside", "secret"]
        if any(h in lower for h in hooks):
            score += 1
            reasons.append("curiosity hook")

        scores.append({"line": clean, "score": score, "max": 8, "reasons": reasons})

    avg = sum(s["score"] for s in scores) / len(scores) if scores else 0
    return {"lines": scores, "average_score": round(avg, 2)}


print("Evaluator ready.")

In [ ]:
# --- Evaluate each provider's output ---
import json

for provider_key, content in results.items():
    display_name = PROVIDERS[provider_key]["display_name"]
    evaluation = score_subject_lines(content)
    print(f"\n--- {display_name} (avg score: {evaluation['average_score']}/8) ---")
    for item in evaluation["lines"]:
        print(f"  [{item['score']}/{item['max']}] {item['line']}")
        print(f"         reasons: {', '.join(item['reasons'])}")

## 8 — Switching Models at Runtime

One of the biggest advantages of the unified pattern is the ability to switch models at runtime
based on cost, latency, or task complexity. The function below demonstrates a simple routing
strategy.

In [ ]:
# --- Model routing strategy ---

ROUTING_TABLE = {
    "subject_line": "google",       # fast + cheap for short copy
    "long_form_blog": "anthropic",   # strong at longer-form content
    "data_analysis": "openai",       # good structured output support
    "brand_review": "anthropic",     # nuanced tone evaluation
    "quick_reply": "google",         # lowest latency
}


def route_task(task_type: str, messages: list, **kwargs):
    """
    Route a martech task to the optimal LLM based on a static routing table.
    In production, this could incorporate real-time latency and cost signals.
    """
    provider_key = ROUTING_TABLE.get(task_type, "openai")
    display_name = PROVIDERS[provider_key]["display_name"]
    print(f"Task '{task_type}' → routed to {display_name}")
    return call_llm(provider_key, messages, **kwargs)


# Demo: route a subject-line task
routed_response = route_task("subject_line", messages)
print(f"\nResult:\n{routed_response.choices[0].message.content}")

## 9 — Cost and Token Tracking

Marketing teams care about spend. Here is a utility that estimates cost from the usage object
returned by each provider.

In [ ]:
# --- Cost estimation utility ---

# Approximate per-1K-token pricing (USD) as of early 2025
PRICING = {
    "openai": {"input": 0.002, "output": 0.008},
    "anthropic": {"input": 0.003, "output": 0.015},
    "google": {"input": 0.0001, "output": 0.0004},
}


def estimate_cost(provider_key: str, prompt_tokens: int, completion_tokens: int) -> float:
    """Estimate USD cost for a single LLM call."""
    rates = PRICING[provider_key]
    return (prompt_tokens / 1000 * rates["input"]) + (completion_tokens / 1000 * rates["output"])


# Estimate cost for a campaign generating 1,000 subject lines
CALLS = 1000
AVG_PROMPT_TOKENS = 150
AVG_COMPLETION_TOKENS = 80

print(f"Estimated cost for {CALLS:,} subject-line generations:\n")
for pkey in PROVIDERS:
    total = CALLS * estimate_cost(pkey, AVG_PROMPT_TOKENS, AVG_COMPLETION_TOKENS)
    print(f"  {PROVIDERS[pkey]['display_name']:30s}  ${total:,.4f}")

## Key Takeaways

1. **Unified client pattern** — The `OpenAI` SDK’s `base_url` parameter lets you target GPT-4.1,
   Claude, and Gemini with identical code. This is the foundation of a provider-agnostic martech
   stack.

2. **Mock-first development** — By gating every call behind `USE_MOCK`, you can develop and test
   pipelines without burning API credits.

3. **Evaluation before deployment** — Even a simple scoring function surfaces real differences
   between models. In production, pair this with human review and A/B testing.

4. **Cost awareness** — Gemini is dramatically cheaper for simple generative tasks; Claude and
   GPT-4.1 justify their price on nuance-heavy work. A routing strategy lets you get the best of
   both worlds.

---

*Next notebook: [02_async_martech.ipynb](./02_async_martech.ipynb) — Async batch email
personalisation for 500 contacts.*